In [1]:
import pandas as pd

df = pd.read_csv("covid19_clinical_trials_normalized.csv")

In [2]:
import re

def parse_age(age_str):
    if pd.isna(age_str):
        return pd.Series([None, None, None])
    
    pop_match = re.search(r'\((.*?)\)', age_str)
    population = pop_match.group(1) if pop_match else age_str.strip()
    
    numeric_part = age_str.split('(')[0].strip()
    
    min_age, max_age = None, None
    
    range_match = re.search(r'(\d+)\s*(Year|Month)s?\s*to\s*(\d+)\s*(Year|Month)s?', numeric_part, re.IGNORECASE)
    older_match = re.search(r'(\d+)\s*(Year|Month)s?\s*and older', numeric_part, re.IGNORECASE)
    upto_match = re.search(r'up to\s*(\d+)\s*(Year|Month|Day)s?', numeric_part, re.IGNORECASE)
    
    def to_years(value, unit):
        value = int(value)
        if unit.lower() == 'month':
            return round(value / 12, 2)
        elif unit.lower() == 'day':
            return round(value / 365, 2)
        return value
    
    if range_match:
        min_age = to_years(range_match.group(1), range_match.group(2))
        max_age = to_years(range_match.group(3), range_match.group(4))
    elif older_match:
        min_age = to_years(older_match.group(1), older_match.group(2))
        max_age = None
    elif upto_match:
        min_age = 0
        max_age = to_years(upto_match.group(1), upto_match.group(2))
    
    return pd.Series([min_age, max_age, population])

df[['min_age', 'max_age', 'population_group']] = df['Age'].apply(parse_age)



In [3]:
# recheck for any remaining unparsed rows
unparsed = df[df['min_age'].isna() & df['max_age'].isna() & ~df['Age'].str.contains('Child, Adult, Older Adult|^Child$|^Adult$|^Older Adult$|Child, Adult$', regex=True, na=False)]
print(unparsed[['Age', 'min_age', 'max_age', 'population_group']])
print(len(unparsed))
# 6 rows with unusual age units (minutes/hours/weeks) or mixed-unit ranges left unparsed — 
# negligible (0.1% of data), documented as known limitation

                                        Age  min_age  max_age population_group
313              up to 15 Minutes   (Child)      NaN      NaN            Child
446          22 Weeks to 28 Weeks   (Child)      NaN      NaN            Child
1802                up to 7 Hours   (Child)      NaN      NaN            Child
3793   31 Days to 18 Years   (Child, Adult)      NaN      NaN     Child, Adult
4560  44 Weeks to 18 Years   (Child, Adult)      NaN      NaN     Child, Adult
4713               up to 72 Hours   (Child)      NaN      NaN            Child
6


In [4]:
date_cols = ['Start Date', 'Primary Completion Date', 'Completion Date', 
             'First Posted', 'Results First Posted', 'Last Update Posted']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(df[date_cols].dtypes)

# trial duration in days
df['trial_duration_days'] = (df['Completion Date'] - df['Start Date']).dt.days

print(df['trial_duration_days'].describe())
print(df[df['trial_duration_days'] < 0][['NCT Number', 'Start Date', 'Completion Date', 'trial_duration_days']])

Start Date                 datetime64[us]
Primary Completion Date    datetime64[us]
Completion Date            datetime64[us]
First Posted               datetime64[us]
Results First Posted       datetime64[us]
Last Update Posted         datetime64[us]
dtype: object
count     4227.000000
mean       402.541992
std        688.088755
min          0.000000
25%        121.000000
50%        266.000000
75%        466.500000
max      29119.000000
Name: trial_duration_days, dtype: float64
Empty DataFrame
Columns: [NCT Number, Start Date, Completion Date, trial_duration_days]
Index: []


In [5]:
print(df.nsmallest(10, 'trial_duration_days')[['NCT Number', 'Start Date', 'Completion Date', 'trial_duration_days', 'Status']])

       NCT Number Start Date Completion Date  trial_duration_days  \
989   NCT04459533 2020-06-01      2020-06-01                  0.0   
1350  NCT04414904 2020-06-10      2020-06-10                  0.0   
1734  NCT04729075 2021-01-19      2021-01-19                  0.0   
2361  NCT04529408 2020-08-26      2020-08-26                  0.0   
2795  NCT04791072 2021-03-01      2021-03-01                  0.0   
2982  NCT04392128 2020-09-02      2020-09-02                  0.0   
3641  NCT04323631 2020-04-30      2020-04-30                  0.0   
3916  NCT04409184 2020-08-14      2020-08-14                  0.0   
3924  NCT04379258 2021-02-08      2021-02-08                  0.0   
4045  NCT04347512 2020-06-02      2020-06-02                  0.0   

                  Status  
989            Completed  
1350           Withdrawn  
1734           Completed  
2361          Recruiting  
2795  Not yet recruiting  
2982           Withdrawn  
3641           Withdrawn  
3916           Withdrawn

In [6]:
df.columns

Index(['NCT Number', 'Title', 'Status', 'Study Results', 'Conditions',
       'Interventions', 'Outcome Measures', 'Sponsor/Collaborators', 'Gender',
       'Age', 'Phases', 'Enrollment', 'Funded Bys', 'Study Type',
       'Study Designs', 'Start Date', 'Primary Completion Date',
       'Completion Date', 'First Posted', 'Results First Posted',
       'Last Update Posted', 'Locations', 'URL', 'min_age', 'max_age',
       'population_group', 'trial_duration_days'],
      dtype='str')

In [7]:
df[df['Completion Date'] == '2099-12-31']

,NCT Number,Title,Status,Study Results,Conditions,Interventions,Outcome Measures,Sponsor/Collaborators,Gender,Age,...,Completion Date,First Posted,Results First Posted,Last Update Posted,Locations,URL,min_age,max_age,population_group,trial_duration_days
4612,NCT04342806,Healthcare Worker Exposure Response and Outcom...,Recruiting,No Results Available,Health Care Worker (HCW)|COVID-19,NaN,Number and percent of participants who enroll ...,Duke University|Patient-Centered Outcomes Rese...,All,"18 Years to 89 Years (Adult, Older Adult)",...,2099-12-31,2020-04-13,NaT,2021-03-30,"Duke Clinical Research Institute, Durham, Nort...",https://ClinicalTrials.gov/show/NCT04342806,18.0,89.0,"Adult, Older Adult",29119.0


In [8]:
# flag the placeholder date instead of treating it as real
df.loc[df['Completion Date'] == '2099-12-31', 'trial_duration_days'] = None
df.loc[df['Completion Date'] == '2099-12-31', 'completion_date_is_placeholder'] = True

In [9]:
df[df['NCT Number'] == 'NCT04342806']

,NCT Number,Title,Status,Study Results,Conditions,Interventions,Outcome Measures,Sponsor/Collaborators,Gender,Age,...,First Posted,Results First Posted,Last Update Posted,Locations,URL,min_age,max_age,population_group,trial_duration_days,completion_date_is_placeholder
4612,NCT04342806,Healthcare Worker Exposure Response and Outcom...,Recruiting,No Results Available,Health Care Worker (HCW)|COVID-19,NaN,Number and percent of participants who enroll ...,Duke University|Patient-Centered Outcomes Rese...,All,"18 Years to 89 Years (Adult, Older Adult)",...,2020-04-13,NaT,2021-03-30,"Duke Clinical Research Institute, Durham, Nort...",https://ClinicalTrials.gov/show/NCT04342806,18.0,89.0,"Adult, Older Adult",NaN,True


In [10]:
df.loc[df['NCT Number'] == 'NCT04342806', ['NCT Number','trial_duration_days','completion_date_is_placeholder']]

,NCT Number,trial_duration_days,completion_date_is_placeholder
4612,NCT04342806,NaN,True


In [11]:
def bucket_enrollment(n):
    if pd.isna(n):
        return None
    elif n < 100:
        return 'Small (<100)'
    elif n < 1000:
        return 'Medium (100-999)'
    elif n < 10000:
        return 'Large (1000-9999)'
    else:
        return 'Very Large (10000+)'

df['enrollment_bucket'] = df['Enrollment'].apply(bucket_enrollment)
print(df['enrollment_bucket'].value_counts())

enrollment_bucket
Medium (100-999)       2705
Small (<100)           1966
Large (1000-9999)       851
Very Large (10000+)     227
Name: count, dtype: int64


In [12]:
df.to_csv('covid19_clinical_trials_feature_engineered.csv', index=False)

print("saved")

saved
